<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/05_텍스트_클러스터링과_토픽_모델링.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 5 서론](#chapter5)
* [Chapter 5-1 아카이브 논문: 계산 및 언어](#chapter5-1)
* [Chapter 5-2 텍스트 클러스터링을 위한 파이프라인](#chapter5-2)
    * [Chapter 5-2-1 문서 임베딩](#chapter5-2-1)
    * [Chapter 5-2-2 임베딩 차원 축소하기](#chapter5-2-2)
    * [Chapter 5-2-3 축소된 임베딩 클러스터링](#chapter5-2-3)
    * [Chapter 5-2-4 클러스터 조사](#chapter5-2-4)
* [Chapter 5-3 텍스트 클러스터링에서 토픽 모델링으로](#chapter5-3)    
    * [Chapter 5-3-1 BERTopic:모듈화된 모델링 프레임워크](#chapter5-3-1)    
    * [Chapter 5-3-2 특수 레고 블록 추가하기](#chapter5-3-2)    
    * [Chapter 5-3-3 텍스트 생성 레고블럭](#chapter5-3-3)    

## Chapter 5 서론 <a class="anchor" id="chapter5"></a>
1. 텍스트 클러스터링은 텍스트의 내용, 의미, 관계를 기반으로 비슷한 것끼리 그룹으로 모으는 것이다.

2. 텍스트 클러스터링은 이상치 찾기, 레이블 할당 속도 향상, 레이블이 잘못 부여된 데이터 찾기 등 다양한 용도로 사용된다.

3. 텍스트 클러스터링은 대규모 텍스트 데이터에서 토픽을 찾는 토픽 모델링 영역에서도 등장한다.
    - 아래의 그림과 같이 토픽을 키워드나 키워드 문구를 사용해 나타낼 수 있다.
    
        ![토픽 예시](./image/05_topic_example.png)


4. 이 장에서 임베딩 모델로 클러스터링을 수행하는 방법과, 토픽 모델링 기법인 BERTTopic를 알아본다.

## Chapter 5-1 아카이브 논문: 계산 및 언어 <a class="anchor" id="chapter5-1"></a>
1. 이 장에서 ArXiv 논문 데이터로 클러스터링과 토픽 모델링 알고리즘을 수행한다. 
    - ArXiv는 과학자들이 연구 논문을 공유하는 온라인 저장소로, 다양한 분야의 논문이 포함되어 있다.
    - 계산 및 언어 분야의 논문을 사용한다.

In [ ]:
# 허깅 페이스에서 Arxiv 논문 데이터 셋을 로드한다.
from datasets import load_dataset

# Arxiv 논문 중 계산 및 언어 분야의 논문을 로드한다.
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

# 데이터를 추출한다.
abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

## Chapter 5-2 텍스트 클러스터링을 위한 파이프라인 <a class="anchor" id="chapter5-2"></a>
1. 일반적으로 클러스터링 방법의 파이프라인은 세 개의 단계로 구성된다.
    - 임베딩 모델을 사용해 입력 문서를 임베딩 벡터로 변환한다.
    - 차원 축소 알고리즘을 사용해 임베딩 벡터의 차원을 줄인다.
    - 클러스터링 알고리즘을 사용해 의미가 비슷한 임베딩 벡터를 그룹으로 모은다.


### Chapter 5-2-1 문서 임베딩 <a class="anchor" id="chapter5-2-1"></a>
1. 임베딩 모델을 사용해 문서를 임베딩으로 변환해야하는데, 의미적으로 비슷한 문서를 찾아야 히기때문에 의미 유사도 작업체 최적화된 임베딩 모델을 선택하는 것이 중요하다.

2. MTEB 리더보드에서 사용할 모델을 선택한다.
    - 클러스터링 작업에 최적화되면서 크기가 작은 모델을 선택한다.

In [ ]:
# thenlper/gte-small 모델을 사용해 문서를 임베딩으로 변환한다.
from sentence_transformers import SentenceTransformer

# 각각의 초록에 대한 임베딩을 만든다.
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(list(abstracts), show_progress_bar=True)

In [ ]:
#  임베딩마다 문서의 의미 표현를 표현한든 384개의 값을 가진다.
embeddings.shape

### Chapter 5-2-2 임베딩 차원 축소하기 <a class="anchor" id="chapter5-2-2"></a>
1. 임베딩 공간이 고차원의 데이터인 경우 해당 데이터에서 의미있는 클라스터를 찾는 것은 어렵다.

2. 차원 축소방법을 사용하면 차원의 크기를 줄이고 더 적은 차원에서 같은 데이터를 표현할 수 있다.
    - 차원 축소의 목표는 고차원 데이터의 구조를 보존하는 저차원의 표현을 찾는 것이다.
    
        ![차원 축소 예시](./image/05_dimensionality_reduction.png)


3. 차원의 축소 시 임의로 차원을 삭제하는 것이 아니라, 데이터의 구조를 보존하는 방식으로 차원을 축소한다.
    - 차원 축소 과정에서 손실 되는 정보와 많은 정보를 유지하는 것 사이에 균형을 맞우는 것이 중요하다.

4. 차원 축소 방법에는 PCA, UMAP등의 방법이 있다.
    - PCA는 선형 차원 축소 방법으로, 데이터의 분산을 최대화하는 방향으로 차원을 축소한다.
    - UMAP는 비선형 차원 축소 방법으로, 데이터의 지역적 구조를 보존하면서 차원을 축소한다.


In [ ]:
# UMAP를 사용해 차원을 5로 축소한다.
#   - 일발적으로 5에서 10 사이의 차원으로 축소하는 것이 잘 작동한다.
from umap import UMAP

umap_model = UMAP(
    n_components=5,  # 축소할 차원의 수
    random_state=42,   # 랜덤 시드
    min_dist=0.0,    # 축소된 공간에서 점들 사이의 최소 거리
    metric="cosine"  # 임베딩 간의 유사도를 측정하는 거리 함수
)

reduced_embeddings = umap_model.fit_transform(embeddings)

### Chapter 5-2-3 축소된 임베딩 클러스터링 <a class="anchor" id="chapter5-2-3"></a>
1. 일반적으로 k-평균과 같은 센트로이드 기반 알고리즘을 사용해 축소된 임베딩을 클러스터링한다.
    - k-평균은 클러스터의 개수를 지정해야 하지만 클러스터의 개수를 지정해야 한다.
    
2. 사전에 클러스터 개수를 알지 못하는 경우 밀도 기반 클러스터링 알고리즘인 HDBSCAN을 사용할 수 있다.
    - 밀도 기반 방법이므로 이상치를 감지할 수 있다.
    
        ![클러스터링 예시](./image/05_clustering_example.png)

In [ ]:
# HDBSCAN를 사용해 축소된 임베딩을 클러스터링 한다.
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(
    min_cluster_size=50,  # 클러스터의 최소 크기
).fit(reduced_embeddings)

clusters = hdbscan_model.labels_

len(set(clusters))

### Chapter 5-2-4 클러스터 조사 <a class="anchor" id="chapter5-2-4"></a>

In [ ]:
# 클러스터 0에서 몇 개의 문서를 확인한다.
import numpy as np

cluster = 0
for index in np.where(clusters == cluster)[0][:3]: # 클러스터 0에서 처음 3개의 문서를 확인한다.
    print(list(abstracts)[index][:300]+"...\n")  # 초록의 처음 300자를 출력한다.

1. 출력된 문서에는 수화를 번역하거나 수화로 번역된 문서를 담고있다.

In [ ]:
# 클러스터 결과를 시각화하기 위해 임베딩을 2차원으로 축소한다.
import pandas as pd

# 384차원에서 2차원으로 축소한다.
reduce_embeddings = UMAP(
    n_components=2,  # 축소할 차원의 수
    random_state=42,   # 랜덤 시드
    min_dist=0.0,    # 축소된 공간에서 점들 사이의 최소 거리
    metric="cosine"  # 임베딩 간의 유사도를 측정하는 거리 함수
).fit_transform(embeddings)

# 데이터 프레임을 만든다.
df = pd.DataFrame(reduce_embeddings, columns=["x", "y"])
df["titles"] = titles
df["clusters"] = [str(c) for c in clusters]

# 정상치와 이상치를 구분한다.
clusters_df = df.loc[df.clusters != "-1",:]
outliers_df = df.loc[df.clusters == "-1",:]

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(outliers_df.x, outliers_df.y, c="grey", alpha=0.05)
plt.scatter(clusters_df.x, clusters_df.y, c=clusters_df.clusters.astype("int"), alpha=0.6,s=2, cmap="tab10")
plt.axis("off")
plt.show()

## Chapter 5-3 텍스트 클러스터링에서 토픽 모델링으로 <a class="anchor" id="chapter5-3"></a>
1. 텍스트 데이터 집합에서 주제나 잠재적인 토픽을 찾는 것을 토픽 모델링이라고 한다.
    - 아래의 그림과 같이 토픽의 의미를 가장 잘 나타내거나 포작하는 키워드나 구를 찾는다.
    
        ![토픽 모델링 예시](./image/05_topic_modeling_example.png)

2. 'sign language'로 토픽을 할당하는 대신 토픽을 설명하는 'sign', 'language', 'translation'과 같은 키워드로 토픽을 설명한다.
    - 토픽에 하나의 레이블을 할당하는 것이아니라 일련의 키워드를 통해 토픽의 의미를 이해한다.

3. 잠재 디리클레 할당같은 전통적인 방법은 각 토픽이 말뭉치 어휘사전에 있는 단어의 확률 분포로 표현된다고 가정한다.
    - 이런 방법은 일반적으로 텍스트 데이터에서 특성을 추출하는데 Bag-of-Words 표현을 사용한다.
    - BoW는 문맥이나 의미를 고려하지 않기 때문에 단어의 의미를 포착하는데 어려움이 있다.

### Chapter 5-3-1 BERTopic:모듈화된 모델링 프레임워크 <a class="anchor" id="chapter5-3-1"></a>
1. 의미적으로 유사한 텍스트 클러스터를 활용하여 다양한 종류의 토픽 표현을 추출하는 모델링 기법이다.
    - 토픽 1의 키워드가 cat, dog, pet이라면, 실제 주제와 거리가 있다. 이 키워드를 살펴보고 사람이 문서의 주체를 유추해야한다.

2. "문서 임베딩 -> 차원 축소 -> 클러 스터링" 단계를 수행하여 의미적으로 유사한 문서 그룹을 만든다.

    ![BERTopic 예시](./image/05_bertopic_example.png)


2. BoW 방식을 사용해 말뭉치의 어휘사전에 있는 단어의 분포를 모델링한다.
    - 문서에서 가장 자주 등장하는 단어의 횟수를 카운트한다.
    - 클러스터 수준에서 등장하는 단어의 횟수를 카운트하도록 수정한다.

3. 'the', 'I'와 같은 불용어는 문서에서 많이 등장하지만 의미는 거의 없기 때문에 낮은 가중치를 할당한다.
    - c-TF-IDF를 사용해 단어의 중요도를 계산한다.

4. C-TF에 각 단어의 IDF를 곱해 단어의 중요도를 계산한다.
    - c-TF-IDF는 클러스터 수준에서 단어의 중요도를 계산하는 방법이다.
    
        ![c-TF-IDF 예시](./image/05_c_tf_idf_example.png)

5. 사이킷런의 CountVectorizer를 사용해 클러스터 수준에서 단어의 중요도를 계산한다.
    - CountVectorizer는 텍스트 데이터를 BoW 표현으로 변환하는 도구이다.

6. 토픽 클러스터링과 토픽 표현 두 단계를 합치면 아래의 그림과 같은 전체 BERTopic 모델링 프레임워크가 된다.
    - 이 파이프라인의 주요 장점은 클러스터링과 토픽 표현 두 단계가 서로 독립적이라는 것이다.
    - 각 파이프라인의 구성 요소를 크게 모듈화 할 수 있다.

    ![BERTopic 모델링 프레임워크](./image/05_bertopic_modeling_framework.png)

7. 아래의 그림과 같이 BERTopic의 모듈화를 통해 핵심 요소들을 쉽게 교체할 수 있다.
    - 예를 들어, 임베딩 모델을 다른 모델로 교체하거나, 차원 축소 방법을 다른 방법으로 교체할 수 있다.
    
    ![BERTopic 모듈화](./image/05_bertopic_modularity.png)

8. BERTopic의 모듈화는 하나의 베이스 모델을 사용해 다양한 사용 사례에 적용할 수 있다.

In [ ]:
#Arxiv 데이터 셋에서 BERTopioc을 실행하기 위해 앞서 정의한 모델과 임베딩을 사용할 수 있다.
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,  # 앞서 정의한 임베딩 모델을 사용한다.
    umap_model=umap_model,            # 앞서 정의한 UMAP 모델을 사용한다.
    hdbscan_model=hdbscan_model,       # 앞서 정의한 HDBSCAN 모델을 사용한다.
    verbose=True                       # 진행 상황을 출력한다.
).fit(abstracts, embeddings)  # BERTopic 모델을 초록과 임베딩으로 학습한다.

9. 각각의 토픽은 Name역에 "_"로 연결된 몇 개의 키워드로 표현된다.
    - 예를 들어, 토픽 1의 Name이 "sign_language_translation"이라면, 이 토픽은 "sign", "language", "translation"과 같은 키워드로 표현된다.
    - 첫 번째 토픽의 레이블은 -1로 할당되는데, 이는 이 토픽이 이상치로 간주된다는 것을 의미한다. -1로 할당된 토픽은 클러스터링 단계에서 클러스터에 할당되지 않은 문서들을 나타낸다.
    - 이상치를 없애려면 k-평균과 같이 이상치를 만들지 않는 알고리즘을 사용하거나 BERTopic의 reduce_outliers 매개변수를 사용할 수 있다.

In [ ]:
topic_model.get_topic_info()

10. 토픽 0은 'speech', 'asr', 'recognition'과 같은 키워드로 표현되며, 이는 이 토픽이 음성 인식과 관련된 문서들을 포함하고 있음을 나타낸다.

In [ ]:
topic_model.get_topic(0)

11. "topic modeling"과 같은 키워드로 토픽을 찾을 수 있다.
    - 유사도가 높은 23 토픽을 조사한다.

In [ ]:
topic_model.find_topics("topic modeling")

In [ ]:
topic_model.get_topic(23)

12. BERTopic의 초록이 이 토픽에 해당하는지 확인한다.

In [ ]:
topic_model.topics_[titles.index('BERTopic: Neural topic modeling with a class-based TF-IDF procedure')]

In [ ]:
# 토픽과 문서의 시각화
fig = topic_model.visualize_documents(
    list(abstracts),  # 시각화할 문서 리스트
    reduced_embeddings=reduced_embeddings, # 앞서 축소한 임베딩을 사용한다.
    width=1200,  # 시각화의 너비
    hide_annotations=True  # 문서의 제목을 숨긴다.
)

fig.update_layout(font=dict(size=6))

### Chapter 5-3-2 특수 레고 블록 추가하기 <a class="anchor" id="chapter5-3-2"></a>
1. BoW로 토픽을 표현하는 방식은 의미나 문맥을 고려하지 않는다.
    - BoW를 사용해 표현을 생성하고 그 다음 더 강력하지만 느린 기법을 사용하여 조절할 수 있다.

2. 초기 결과의 순서를 재정정하는 것을 리랭커라고 한다.

3. 리랭커 모델을 BERTopic에서는 표현모델이라고 부른다.

4. BERTopic에 준비된 다양한 표현 블록을 사용해 미세 튜닝을 할 수 있다.
    
    ![BERTopic 표현 블록](./image/05_bertopic_representation_blocks.png)


In [ ]:
# 표현 모델을 사용할 때와 그렇지 않을 때를 비교하기 위해 원본 토픽 표현을 복사한다.
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
def topic_differences(model, original_topics, nr_topics=5):
    """두 모델의 토픽 표현 차이를 보여줍니다"""
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # 모델과 토픽마다 상위 5개 단어를 추출합니다.
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]

    return df

5. KeyBERTInspired는 BERTopic의 표현 모델로 사용할 수 있는 모델 중 하나이다.
    - 문서 c-TF-IDF 값과 토픽의 c-TF-IDF 값 사이의 유사도를 계산하여 토픽마다 가장 대표적인 문서를 추출한다.
    - 아래의 그림과 같이 토픽마다 평균적인 문서 임베딩을 계산하고 후보 키워드 임베딩을 비교하여 키워드의 순위를 계산한다.
    
        ![KeyBERTInspired 예시](./image/05_keybert_inspired_example.png) 

In [ ]:
from bertopic.representation import KeyBERTInspired
import pandas as pd

# KeyBERTInspired를 사용해 토픽 표현을 업데이트합니다.
representation_model = KeyBERTInspired()
topic_model.update_topics(abstracts, representation_model=representation_model)

# 토픽의 차이를 보여줍니다.
topic_differences(topic_model, original_topics)

6. 원본 모델에 비해 업데이트된 모델의 결과 토픽을 이해하기 쉽다.

7. 원본 모델의 nmt 같은 약자가는 표현모델이 적절하게 표현할 수 없어 삭제되었는데 이는 도메인 전문가에게는 유용한 정보이다.

8. MMR을 사용하면 토픽 표현을 다양화 할 수 있다.
    - 비교하려는 무넛와 관련 있지만 서로 다른 키워드 집합을 찾으려고 노력한다.
    - 키워드가 얼마나 다양해야 하는지 지정 후 키워드를 반복적으로 계산해 추가한다.
    - 중복된 단어를 제거하고 토픽 표현에 새롭게 기여하는 단어만 유지한다.

In [ ]:
from bertopic.representation import MaximalMarginalRelevance

# MaximalMarginalRelevance를 사용해 토픽 표현을 업데이트합니다.
representation_model = MaximalMarginalRelevance(diversity=0.5)
topic_model.update_topics(abstracts, representation_model=representation_model)

# 토픽의 차이를 보여줍니다.
topic_differences(topic_model, original_topics)


### Chapter 5-3-3 텍스트 생성 레고블럭 <a class="anchor" id="chapter5-3-3"></a>
1. 생성 모델을 사용해 토픽을 위한 레이블을 생성할 수 있다.
    - 표현 모델에 키워드와 문서를 기반으로 짧은 레이블을 생성하도록 요청한다.

2. 프롬프트에는 두 가지 주요 요소가 포함된다.
    - [DOCUMENTS]: 토픽을 가장 잘 표현하는 적은 개수 문서가 삽입되면, 해당 토픽과 c-TF-IDF 값이 가장 높은 문서가 삽입된다.
    - [KEYWORDS]: 이 키워드는 c-TF-IDF나 다른 표현 모델을 통해서 생성할 수 있다.


3. 수백만 개가 될 수 있는 문서마다 적용하는 것이 아니라 수백 개 정도의 토픽마다 생성 모델을 한 번 사용한다.

    ![텍스트 생성 레고블럭 예시](./image/05_text_generation_representation_block_example.png)

In [ ]:
from transformers import pipeline
from bertopic.representation import TextGeneration

prompt = """I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the documents and keywords, what is this topic about?"""

# Flan-T5를 사용해 토픽 표현을 업데이트합니다.
generator = pipeline('text-generation', model='google/flan-t5-small')
representation_model = TextGeneration(
    generator, prompt=prompt, doc_length=50, tokenizer="whitespace"
)
topic_model.update_topics(abstracts, representation_model=representation_model)

# 토픽의 차이를 보여줍니다.
topic_differences(topic_model, original_topics)